# Notebook 04 — Results vs Paper

**Purpose:** Compare reproduced H.BLSTM results against the paper's Table 2/3,
and compute the RQ1 metrics (MASE, Directional Accuracy).

Sections:
1. Bootstrap + load results
2. Reproduction comparison table (MAE, RMSE, MAPE vs paper)
3. MASE analysis — does H.BLSTM show skill over naive persistence?
4. Directional accuracy
5. Inference latency distribution
6. Retrain cost summary
7. Export thesis-ready tables


## 0. Bootstrap

In [ ]:
import os, sys, subprocess
from pathlib import Path

GITHUB_URL = "https://github.com/stephugm/incremental-learning.git"
FORCE_DOWNLOAD = False

def detect_env():
    if "google.colab" in sys.modules: return "colab"
    if os.path.exists("/kaggle/input"): return "kaggle"
    return "local"

ENV = detect_env()
print(f"Environment: {ENV}")

if ENV in ("colab", "kaggle"):
    REPO = (Path("/content/incremental-learning") if ENV == "colab"
            else Path("/kaggle/working/incremental-learning"))
    if not REPO.exists():
        subprocess.run(["git", "clone", GITHUB_URL, str(REPO)], check=True)

    # ── Step 1: install requirements (tvkit hanya untuk lokal / NB05) ────
    subprocess.run(["pip", "install", "-q", "-r",
                    str(REPO / "requirements.txt")], check=False)

    # ── Step 2: salin parquet hasil NB05 (tvkit) dari Kaggle Dataset ──────
    import shutil
    KAGGLE_DS = Path("/kaggle/input/datasets/stephanus88/hblstm-index-15m-raw")
    RAW_DIR = Path("/kaggle/working/thesis_data/raw")
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    if KAGGLE_DS.exists():
        for _f in KAGGLE_DS.glob("*.parquet"):
            if not (RAW_DIR / _f.name).exists():
                shutil.copy(_f, RAW_DIR / _f.name)
            print(f"Dataset : {_f.name}")
        print("tvkit parquet : OK  -> target 6000 bar dapat dicapai")
    else:
        print(f"tvkit parquet : {KAGGLE_DS} TIDAK ADA (Add Data di sidebar Kaggle)")
        print("  -> fallback yfinance, hanya ~1500 bar (limit 60 hari)")
        print("  -> n_bars=6000 di config TIDAK akan tercapai via yfinance")

else:
    REPO = Path().resolve()
    while not (REPO / "configs" / "config.yaml").exists() and REPO != REPO.parent:
        REPO = REPO.parent

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print(f"Repo root: {REPO}")
# autoreload: best-effort, skip gracefully on Python 3.12+ if imp removed
try:
    _ip = get_ipython()  # noqa: F821
    _ip.run_line_magic('load_ext', 'autoreload')
    _ip.run_line_magic('autoreload', '2')
except Exception as _ar_e:
    print(f'autoreload skipped: {_ar_e}')

In [ ]:
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.paths import Paths
from src.metrics import (
    evaluate, compare_to_paper, summarise_runs,
    mae, rmse, mape, mase, directional_accuracy, naive_persistence
)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

paths = Paths()
with open(paths.configs / "config.yaml") as f:
    cfg = yaml.safe_load(f)

INDICES      = cfg["data"]["indices"]
FEATURE_SETS = list(cfg["data"]["feature_sets"].keys())
SEEDS        = cfg["training"]["seeds"]

## 1. Load all results

In [ ]:
results_path = paths.results / "hblstm_all_results.json"
assert results_path.exists(), (
    f"Results not found at {results_path}. Run Notebook 03 first."
)

with open(results_path) as f:
    all_results = json.load(f)

df_results = pd.DataFrame(all_results)
print(f"Loaded {len(df_results)} runs")
df_results.head()

## 2. Reproduction comparison table

Compare MAE, RMSE, MAPE (mean ± std across 3 seeds) against paper's Table 2/3.

In [ ]:
print("=" * 70)
print("MULTIVARIATE — H.BLSTM reproduction vs Kalra et al. (2024) Table 2/3")
print("=" * 70)
multi_results = [r for r in all_results if r["feature_set"] == "multivariate"]
cmp_df = compare_to_paper(multi_results, feature_set="multivariate")

In [ ]:
print("=" * 70)
print("UNIVARIATE — H.BLSTM (U_H.BLSTM) reproduction")
print("=" * 70)
uni_results = [r for r in all_results if r["feature_set"] == "univariate"]
cmp_uni_df = compare_to_paper(uni_results, feature_set="univariate")

## 3. MASE analysis (RQ1)

MASE < 1.0 → model has skill over naive persistence  
MASE ≈ 1.0 → model is equivalent to naive persistence (no skill)  
MASE > 1.0 → model is worse than naive persistence

Hypothesis H1: *H.BLSTM's MASE will be ≥ 0.90 (near 1.0), indicating
no meaningful skill over naive persistence.*

In [ ]:
# Recompute MASE directly from saved per-bar predictions for robustness
mase_rows = []

for idx_cfg in INDICES:
    sym = idx_cfg["symbol"]
    for fset in FEATURE_SETS:
        for seed in SEEDS:
            preds_path = paths.preds_path(sym, fset, seed)
            if not preds_path.exists():
                continue
            df_p = pd.read_parquet(preds_path)
            y_t = df_p["y_true_raw"].values
            y_p = df_p["y_pred_raw"].values

            # Naive persistence
            y_naive = naive_persistence(y_t)
            y_t_aligned = y_t[1:]     # drop first (NaN naive)
            y_p_aligned = y_p[1:]
            y_n_aligned = y_naive[1:]

            m = mase(y_t_aligned, y_p_aligned, y_naive=y_n_aligned)
            mase_rows.append({
                "symbol": sym, "feature_set": fset, "seed": seed, "mase": m
            })

mase_df = pd.DataFrame(mase_rows)
mase_summary = mase_df.groupby(["symbol", "feature_set"])["mase"].agg(["mean", "std"])
print("MASE summary (mean ± std across seeds):")
print(mase_summary.to_string())
print()
print(f"H1 threshold: MASE >= 0.90 (near 1.0 = no skill)")
print(f"H1 falsified if: MASE < 0.85 (genuine skill)")

In [ ]:
# Visualise MASE by symbol and feature set
fig, ax = plt.subplots(figsize=(10, 5))

symbols = [i["symbol"] for i in INDICES]
x = np.arange(len(symbols))
w = 0.35

for j, fset in enumerate(FEATURE_SETS):
    means = []
    stds  = []
    for sym in symbols:
        sub = mase_df[(mase_df["symbol"] == sym) & (mase_df["feature_set"] == fset)]["mase"]
        means.append(sub.mean() if len(sub) > 0 else np.nan)
        stds.append(sub.std()   if len(sub) > 1 else 0.0)
    ax.bar(x + j * w, means, w, label=fset, yerr=stds, capsize=4, alpha=0.8)

ax.axhline(1.0, color="red", ls="--", lw=1.2, label="MASE = 1.0 (no skill)")
ax.axhline(0.85, color="green", ls=":", lw=1.0, label="MASE = 0.85 (H1 threshold)")
ax.set_xticks(x + w / 2)
ax.set_xticklabels(symbols)
ax.set_ylabel("MASE")
ax.set_title("MASE by index and feature set (lower = more skill)")
ax.legend()
plt.tight_layout()
fig.savefig(paths.figures / "mase_comparison.png", dpi=150)
plt.show()

## 4. Directional Accuracy (DA)

In [ ]:
da_rows = []
for idx_cfg in INDICES:
    sym = idx_cfg["symbol"]
    for fset in FEATURE_SETS:
        for seed in SEEDS:
            preds_path = paths.preds_path(sym, fset, seed)
            if not preds_path.exists(): continue
            df_p = pd.read_parquet(preds_path)
            y_t = df_p["y_true_raw"].values
            y_p = df_p["y_pred_raw"].values
            da  = directional_accuracy(y_t, y_p)
            da_rows.append({"symbol": sym, "feature_set": fset, "seed": seed, "da": da})

da_df = pd.DataFrame(da_rows)
da_summary = da_df.groupby(["symbol", "feature_set"])["da"].agg(["mean", "std"])
print("Directional Accuracy (mean ± std across seeds):")
print(da_summary.to_string())
print()
print("DA = 0.50 → random (coin-flip)")
print("DA > 0.50 → some directional predictive power")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for j, fset in enumerate(FEATURE_SETS):
    means = []
    stds  = []
    for sym in symbols:
        sub = da_df[(da_df["symbol"] == sym) & (da_df["feature_set"] == fset)]["da"]
        means.append(sub.mean() if len(sub) > 0 else np.nan)
        stds.append(sub.std()   if len(sub) > 1 else 0.0)
    ax.bar(x + j * w, means, w, label=fset, yerr=stds, capsize=4, alpha=0.8)

ax.axhline(0.5, color="red", ls="--", lw=1.2, label="DA = 0.50 (random)")
ax.set_xticks(x + w / 2)
ax.set_xticklabels(symbols)
ax.set_ylabel("Directional Accuracy")
ax.set_ylim(0.4, 0.7)
ax.set_title("Directional Accuracy by index and feature set (higher = more skill)")
ax.legend()
plt.tight_layout()
fig.savefig(paths.figures / "da_comparison.png", dpi=150)
plt.show()

## 5. Inference latency distribution

In [ ]:
# Load predictions for SPX multivariate seed 0 (demo run)
DEMO_SYM  = "SPX"
DEMO_FSET = "multivariate"
DEMO_SEED = SEEDS[0]

preds_path = paths.preds_path(DEMO_SYM, DEMO_FSET, DEMO_SEED)
if preds_path.exists():
    df_p = pd.read_parquet(preds_path)
    lat = df_p["infer_ms"]
    print(f"Inference latency statistics (ms):")
    print(lat.describe().to_string())
    print(f"\nPaper reports: avg 2s (= 2000 ms)")
    print(f"Our avg: {lat.mean():.1f} ms  (on {str(DEVICE)})")

    fig, ax = plt.subplots(figsize=(8, 3))
    lat.clip(upper=lat.quantile(0.99)).plot.hist(ax=ax, bins=60, density=True)
    ax.axvline(lat.mean(), color="red", ls="--", lw=1.2, label=f"Mean = {lat.mean():.1f} ms")
    ax.axvline(2000, color="orange", ls=":", lw=1.2, label="Paper: 2000 ms")
    ax.set_xlabel("Inference latency (ms)")
    ax.set_title(f"{DEMO_SYM} {DEMO_FSET} — per-bar inference latency")
    ax.legend()
    plt.tight_layout()
    fig.savefig(paths.figures / f"{DEMO_SYM}_latency.png", dpi=150)
    plt.show()
else:
    print(f"No predictions found at {preds_path}. Run Notebook 03 first.")

## 6. Retrain cost summary (preview of RQ3)

In [ ]:
# Retrain events are logged in each preds parquet
cost_rows = []
for idx_cfg in INDICES:
    sym = idx_cfg["symbol"]
    for fset in FEATURE_SETS:
        for seed in SEEDS:
            pp = paths.preds_path(sym, fset, seed)
            if not pp.exists(): continue
            df_p = pd.read_parquet(pp)
            n_retrains = df_p["retrain_triggered"].sum()
            cost_rows.append({
                "symbol": sym, "feature_set": fset, "seed": seed,
                "n_retrains": int(n_retrains),
                "n_sessions": df_p["session_id"].nunique(),
            })

if cost_rows:
    cost_df = pd.DataFrame(cost_rows)
    cost_summary = cost_df.groupby(["symbol", "feature_set"]).agg(
        sessions=("n_sessions", "mean"),
        retrains=("n_retrains", "mean"),
    )
    print("Retrain summary (fixed end-of-session policy):")
    print(cost_summary.to_string())
    print("\n(Full cost metrics — wall-clock, peak memory — will be in RQ3 notebook)")
else:
    print("No prediction files found.")

## 7. Export thesis-ready tables

In [ ]:
# Table 1: full reproduction comparison
cmp_path = paths.results / "table_reproduction.csv"
cmp_df.to_csv(cmp_path)
print(f"Saved reproduction table → {cmp_path}")

# Table 2: MASE + DA summary
rq1_df = mase_summary.join(da_summary, rsuffix="_da").rename(
    columns={"mean": "mase_mean", "std": "mase_std",
             "mean_da": "da_mean", "std_da": "da_std"}
)
rq1_path = paths.results / "table_rq1_mase_da.csv"
rq1_df.to_csv(rq1_path)
print(f"Saved RQ1 table → {rq1_path}")

# Full summary
summary = summarise_runs(all_results)
summary_path = paths.results / "full_metrics_summary.csv"
summary.to_csv(summary_path, index=False)
print(f"Saved full summary → {summary_path}")

## 8. Interpretation guide

Use this cell to write your thesis narrative once results are populated.

In [ ]:
print("Interpretation checklist:")
print()
print("1. MAPE comparison:")
print("   □ Is our MAPE in the same order of magnitude as the paper (0.001)?")
print("   □ If not, document the gap as a finding (discrepancy in data, hyperparams, etc.)")
print()
print("2. MASE (RQ1):")
print("   □ MASE ≥ 0.90 → H1 supported (no meaningful skill over naive persistence)")
print("   □ MASE < 0.85 → H1 falsified (genuine skill — good for the base paper)")
print()
print("3. Directional Accuracy (RQ1):")
print("   □ DA ≈ 0.50 → coin-flip, no directional skill")
print("   □ DA > 0.55 (multivariate) → some directional signal")
print()
print("4. Uni vs Multi comparison:")
print("   □ Does multivariate consistently beat univariate?")
print("   □ If margin is small → supports EMA5-as-smoothing hypothesis (RQ2)")